## How to create pyAPES forcing file?

Samuli Launiainen 25.2.2026

### We create here pyAPES forcing for Soroe beech forest ICOS site 

- use ICOS data release


In [ ]:
# setting path
import sys
import os
from dotenv import load_dotenv

load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')
# Fallback to workspace path if environment variable not set
if pyAPES_main_folder is None:
    pyAPES_main_folder = r'c:\Repositories\pyAPES_main'

sys.path.append(pyAPES_main_folder)

# import modules
from scipy import stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
print(f"pyAPES_main_folder: {pyAPES_main_folder}")
print(f"Path exists: {os.path.exists(pyAPES_main_folder)}")

### Read Degerö ICOS L2 data

- prepare model forcing
- prepare model evaluation dataset

https://fluxnet.org/data/fluxnet2015-dataset/variables-quick-start-guide/

In [ ]:
ffile = r'c:\Data\Fluxnet2015\DK-Sor\FLX_DK-Sor_FLUXNET2015_FULLSET_HH_1996-2020_beta-3.csv'
#mfile =r'c:\Data\Fluxnet2015\DK-Sor\FLX_DK-Sor_FLUXNET2015_ERAI_HH_1989-2020_beta-3.csv'

raw = pd.read_csv(ffile, sep=',', na_values=-9999)
t = pd.to_datetime(raw['TIMESTAMP_END'].astype(str))
raw.index = t

# select 2021->
raw = raw[raw.index > '2015-01-01 00:00:00']



In [ ]:
print(raw.columns.tolist())

In [ ]:
cols = ['WD', 'USTAR', 'RH', 'NETRAD', 'PPFD_IN', 'PPFD_DIF', 'PPFD_OUT', 'SW_OUT', 'LW_OUT', 'CO2_F_MDS', 'TS_F_MDS_1', 'TS_F_MDS_2', 'TS_F_MDS_3', 'TS_F_MDS_4', 'TS_F_MDS_5', 'TS_F_MDS_6',
        'SWC_F_MDS_1', 'SWC_F_MDS_2', 'SWC_F_MDS_3', 'SWC_F_MDS_4', 'SWC_F_MDS_5',  'G_F_MDS', 'G_F_MDS_QC', 'LE_F_MDS', 'LE_F_MDS_QC', 'H_F_MDS', 'H_F_MDS_QC', ]

### Create output dataframe

In [ ]:
plt.plot(raw[['GPP_NT_VUT_REF', 'GPP_DT_VUT_REF' ]])
plt.plot(raw[['RECO_NT_VUT_REF', 'RECO_DT_VUT_REF' ]])

In [ ]:
plt.plot(raw[['SW_IN_F', 'SW_OUT']])
plt.plot(raw[['LW_IN_F', 'LW_OUT']])

In [ ]:
forcing_outfile = os.path.join(pyAPES_main_folder, r'forcing/Soroe/Soroe_forcing_2015-2020.dat')
met_outfile = os.path.join(pyAPES_main_folder, r'forcing/Soroe/Soroe_meteo_2015-2020.dat') 
fluxes_outfile = os.path.join(pyAPES_main_folder, r'forcing/Soroe/Soroe_EC_2015-2020.dat') 

fcols = ['year', 'month', 'day', 'hour', 'minute', 'doy',
         'Prec','P','Tair', 'U', 'Ustar', 'H2O', 'CO2', 'RH', 
         'Zen', 'LWin', 'diffPar', 'dirPar', 'diffNir', 'dirNir',
         'DDsum', 'X', 'Tdaily', 'Tsoil', 'SWC']

forcing = pd.DataFrame(data=None, columns=fcols, index=raw.index)

forcing['year'] = raw.index.year
forcing['month'] = raw.index.month
forcing['day'] = raw.index.day
forcing['hour'] = raw.index.hour
forcing['minute'] = raw.index.minute
forcing['doy'] = raw.index.dayofyear

### Radiation variables


In [ ]:
from pyAPES.microclimate.radiation import solar_angles, compute_clouds_rad
from pyAPES.microclimate.micromet import e_sat

from pyAPES.utils.constants import DEG_TO_RAD, DEG_TO_KELVIN, PAR_TO_UMOL

fpar = 0.40 # ratio of Par/Global

Observed Par as function of Global radiation

In [ ]:
x = raw['SW_IN_F'].values
y = raw['PPFD_IN'].values / PAR_TO_UMOL
ix = np.where((x > 0) & (y >0))[0]

p = np.polyfit(x[ix], y[ix], 1)
fit = np.polyval(p, [0.0, max(x[ix])])

fig, ax = plt.subplots(1,1, figsize=(3,3))
ax.plot(x[ix], y[ix], 'ro')
ax.plot([0.0, max(x[ix])], fit, 'k-', label='%.2f' %p[0])
ax.legend()
ax.set_xlabel('SW in'); ax.set_ylabel('PAR')


In [ ]:
%matplotlib inline

# compute solar angles
lat = 55.486 # N
lon = 11.644 # E
timeoffset = 1.15 # accounts for +1 UTC and 15min correction due to timestep being ENDTIME of 30min period

jday = forcing.index.dayofyear + forcing.index.hour / 24 + forcing.index.minute / (24*60)
jday = jday.values

# global radiation, air temperature, rh and air pressure are gap-filled in FluxNet2015 datarelease
rg = raw['SW_IN_F'].values # Wm-2
ta = raw['TA_F'].values # degC
es, _ = e_sat(ta) # saturation vapor pressure [Pa]

# rh is gappy, FluxNet2015 variable 'VPD_F' [hPa] is gap-filled
#rh = raw['RH'].values
#rh = np.maximum(1.0, np.minimum(rh, 100.0)) # make rh be in realistic range
#h2o = rh / 100 *es # vapor pressure [Pa]

vpd = 100 * raw['VPD_F'].values #[Pa]
h2o = es - vpd
rh = h2o / es
#rh = np.maximum(1.0, np.minimum(rh, 100.0)) # make rh be in realistic range
#h2o = rh / 100 *es # vapor pressure [Pa]

# compute solar zenit angle and estimate diffuse fraction
zen, azim, decl, sunrise, sunset, daylength = solar_angles(lat, lon, jday, timezone=timeoffset)
elev_angle = 90 - zen / DEG_TO_RAD

fcloud, fdiff, emi_sky = compute_clouds_rad(jday, zen, rg, h2o, ta)


t = raw.index

fig, ax = plt.subplots(3,1)
ax[0].plot(t, raw['SW_IN_POT'].values, '-', alpha=0.5, label='SW in pot.'), 
ax[0].legend()
ax[0].set_ylabel('Wm-2')
axb = ax[0].twinx()
axb.plot(t, elev_angle, 'r-', alpha=0.5, label='solar elev. (deg)')
axb.legend()

ax[1].plot(t, rg, '-', alpha=0.5, label='Rg')
ax[1].plot(t, fdiff * rg, 'r-', alpha=0.5, label='diff Rg')
ax[1].legend()
ax[1].set_ylabel('Wm-2')

ax[2].plot(t, fcloud, '-')
ax[2].set_ylabel('cloud fraction (-)')

# update forcing

forcing['Tair'] = ta
forcing['Prec'] = raw['P_F'].values
forcing['P'] = 1000 * raw['PA_F'].values # Pa
forcing['CO2'] = 390.0
forcing['H2O'] = 1000 * h2o / forcing['P'].values # mmol/mol = ppth
forcing['U'] = raw['WS_F'].values
forcing['RH'] = rh * 100

forcing['Zen'] = zen
forcing['LWin'] = raw['LW_IN_F'].values
forcing['dirPar'] = fpar * (1 - fdiff) * rg
forcing['diffPar'] = fpar *  fdiff * rg
forcing['dirNir'] = (1 - fpar) * (1 - fdiff) * rg
forcing['diffNir'] = (1 - fpar) *  fdiff * rg

# soil temperature and moisture
ts = raw['TS_F_MDS_1'].interpolate('linear').values
swc = raw['SWC_F_MDS_2'].interpolate('linear').values
forcing['Tsoil'] = ts
forcing['SWC'] = swc

### wind speed and friction velocity check


In [ ]:
def filter_binned(x, y, bins=20, lim=98.5):
    bin_lim, bin_edges, binnumber = stats.binned_statistic(x, y, statistic=lambda y: np.nanpercentile(y, lim), bins=bins)
    
    xl = bin_edges[0]
    xu = bin_edges[1]
    
    yout = y.copy()
    for j in range(0, len(bin_lim)):
        xl = bin_edges[j]
        xu = bin_edges[j+1]
        #print(j, xl, xu, bin_lim[j])
        ix = np.where((x>=xl) & (x<=xu))[0]
        yout[ix] = np.minimum(y[ix], bin_lim[j])

    return yout, bin_lim, bin_edges, binnumber

Friction velocity - fill gaps by linear interpolation

In [ ]:
gap_fill = 'lin_interp' # or linear interpolation 'lin_interp'

fig, ax = plt.subplots(1,2, figsize=(8,4))

u = raw['WS_F'].copy()
ust_r = raw['USTAR'].copy()

# screen clear outliers
ix = np.where((u > 15 )| (ust_r > 0.8))[0]
u[ix] = np.nan
ust_r[ix] = np.nan
ix = np.where(u > 0)[0]
ax[0].hist(u[ix], 100)
#ax[0].set_xlim([0,30])

ix = np.where(ust_r > 0)[0]
ax[1].hist(ust_r[ix], 100)
ax[1].set_xlim([0,3])

fig, ax = plt.subplots(1,2, figsize=(8,4))

ax[0].plot(u, ust_r, 'k.')

ax[1].plot(u, 'k-')
ax[1].plot(ust_r, 'r.')

#--- remove outliers in WS bins

# store indices of finite data
ix = np.where((np.isfinite(u))| (np.isfinite(ust_r)))[0]
u0 = u[ix]
ust_r0 = ust_r[ix]

y, lims, binedges, binmeans = filter_binned(u0, ust_r0, bins=50, lim=99.0)

# copy back to right place in full vector
ust_r[ix] = y

ax[0].plot(u0, y, 'r.', alpha=0.1)

ix = np.where(np.isfinite(u))[0]
x = np.arange(0, len(u))
uf = np.interp(x, ix, u[ix])

# interpolate linearly
ix = np.where(np.isfinite(ust_r))[0]
yf = np.interp(x, ix, ust_r[ix])

if gap_fill == 'lin_interp':
    forcing['Ustar'] = yf
    forcing['U'] = uf
elif gap_fill == 'era5':
    forcing['U'] = u
    forcing['U'] = forcing['U'].fillna(raw['WS_ERA'])
    forcing['Ustar'] = yf


Degree-day sum and phenology modifier
- from *pyAPES.planttype.phenology*

In [ ]:
def degree_days(Tdaily, jday, Tbase=+5.0):
    N = len(Tdaily)
    y = np.zeros(N)
    
    for k in range(1, N):
        if jday[k] == 1:
            y[k] = 0.0
        else:
            y[k] = y[k-1] + np.max([Tdaily[k] - Tbase, 0.0])
    return y

def pheno_state(T, tau, x0 = 0.0):
    X = np.zeros(len(T)) + np.NaN
    X[0] = x0
    for k in range(1, len(X)):
        X[k] = X[k-1] + 1.0 / tau * (T[k] - X[k-1])  # degC
    return X

In [ ]:
# Functions work on daily mean temperature
Tday = raw['TA_F'].resample('1D').mean()

doy = Tday.index.day_of_year

DDsum = degree_days(Tday.values, doy)
X = pheno_state(Tday.values, tau=8.0, x0=Tday[0])

tmp = pd.DataFrame(Tday)
tmp['DDsum'] = DDsum
tmp['X'] = X

# re-index and interpolate by forward-filling, except 1st day
tmp = tmp.reindex(index=raw.index)
tmp = tmp.ffill()
tmp = tmp.bfill()

# update forcing
forcing['Tdaily'] = tmp['TA_F'].values
forcing['DDsum'] = tmp['DDsum'].values
forcing['X'] = tmp['X'].values

### Plot forging variables & save to file

In [ ]:
%matplotlib inline
forcing = forcing[(forcing.index >= '2015-01-01')] 

for k in forcing.columns:
    plt.figure()
    plt.plot(forcing[k], label=k)
    gaps = len(np.where(np.isnan(forcing[k]))[0])
    plt.title(k + ' gaps: %d' %gaps)
    #plt.xlim(['2011-01-01', '2013-01-01'])
    plt.legend()

In [ ]:
forcing[['year', 'month', 'day', 'hour', 'minute', 'doy']].astype(int)
forcing.to_csv(forcing_outfile, sep=';', float_format='%.3f', index=False)
#fluxes.to_csv(fluxes_outfile, sep=';', float_format='%.3f', index=False)
plt.close('all')

### Extract Fluxnet2015 data subset for model evaluation


In [ ]:
fcols = [# radiation variables
         'NETRAD', 'SW_IN_F', 'PPFD_IN', 'PPFD_OUT', 'SW_OUT', 'LW_IN_F', 'LW_OUT',
         # energy fluxes
         'H_F_MDS', 'LE_F_MDS', 'G_F_MDS',
         # NEE, GPP, RECO
         'NEE_VUT_REF', 'GPP_NT_VUT_REF', 'RECO_NT_VUT_REF', 
         'GPP_DT_VUT_REF', 'RECO_DT_VUT_REF', 
         'TA_F', 'TS_F_MDS_1', 'TS_F_MDS_2', 'TS_F_MDS_3', 'TS_F_MDS_4', 'TS_F_MDS_5',
         'SWC_F_MDS_1', 'SWC_F_MDS_2', 'SWC_F_MDS_3', 'SWC_F_MDS_4', 'SWC_F_MDS_5', 
         'WS_F', 'WD', 'NIGHT', 'P_F', 
         # quality flags
         'SW_IN_F_QC', 'LW_IN_F_QC', 'H_F_MDS_QC', 'LE_F_MDS_QC', 'G_F_MDS_QC', 'NEE_VUT_REF_QC', 'VPD_F_QC', 'TA_F_QC', 'P_F_QC'
         ]

flx = raw[fcols]
flx['NETRAD'] = raw['SW_IN_F'] - raw['SW_OUT'] + raw['LW_IN_F'] - raw['LW_OUT']

elev_angle = 90.0 - forcing['Zen'] / DEG_TO_RAD
ix = np.where(elev_angle <10.0)[0]
sw_alb = flx['SW_OUT'] / (flx['SW_IN_F'] + 0.01)
sw_alb[ix] = np.NaN
sw_alb[(sw_alb > 1.0) | (sw_alb < 0)] = np.NaN

par_alb = flx['PPFD_OUT'] / (flx['PPFD_IN'] + 0.01)
par_alb[ix] = np.NaN
par_alb[(par_alb > 1.0) | (par_alb < 0)] = np.NaN

flx.insert(7,'SW_ALB', sw_alb.values)
flx.insert(8,'PPDF_ALB', par_alb.values)
flx.insert(0,'year', flx.index.year)
flx.insert(1,'month', flx.index.month)
flx.insert(2,'day', flx.index.day)
flx.insert(3,'hour', flx.index.hour)
flx.insert(4,'minute', flx.index.minute)
flx.insert(5,'doy', flx.index.dayofyear)

flx = flx[(flx.index >='2015-01-01')]

In [ ]:
flx.columns

for k in flx.columns:
    plt.figure()
    plt.plot(flx[k], label=k)
    gaps = len(np.where(np.isnan(flx[k]))[0])
    plt.title(k + ' gaps: %d' %gaps)
    #plt.xlim(['2011-01-01', '2013-01-01'])
    plt.legend()

In [ ]:
flx[['year', 'month', 'day', 'hour', 'minute', 'doy']].astype(int)
flx.to_csv(fluxes_outfile, sep=';', float_format='%.3f', index=False, na_rep='NaN')
plt.close('all')